# Mizan · Baseline 2: zero-shot open-source LLM

Can a general-purpose open-source LLM classify Arabic banking requests **with no training at all**?

- Model: **Qwen2.5-1.5B-Instruct** (open weights, Apache-2.0, runs locally, so no data leaves the machine)
- Task: choose one of 77 banking intents for each of the 370 test queries (179 MSA, 191 Palestinian dialect)
- Same PII redaction as the rest of Mizan; same scoring code (`mizan.evaluate`)

**Runtime:** Runtime → Change runtime type → **T4 GPU**. Takes about 5–10 minutes.

## 1. Get the code and data

In [ ]:
REPO_URL = "https://github.com/zainabali18/Mizan.git"

!git clone -q $REPO_URL mizan_repo
%cd mizan_repo
!pip install -q -r requirements.txt transformers accelerate scikit-learn
!python -m mizan.data > /dev/null && echo "processed data ready"

## 2. Load the model

In [ ]:
import json, re, time
import pandas as pd
import torch
from transformers import AutoModelForCausalLM, AutoTokenizer

MODEL_ID = "Qwen/Qwen2.5-1.5B-Instruct"
tok = AutoTokenizer.from_pretrained(MODEL_ID)
tok.padding_side = "left"   # needed for batched generation
model = AutoModelForCausalLM.from_pretrained(MODEL_ID, torch_dtype=torch.float16, device_map="auto")
model.eval()

n_params = sum(p.numel() for p in model.parameters())
SIZE_MB = n_params * 2 / 1e6   # fp16 = 2 bytes per parameter
print(f"{n_params/1e9:.2f}B parameters, ~{SIZE_MB:,.0f} MB in fp16, device={model.device}")

## 3. Build the prompt

The model sees the 77 intent names (in Arabic, numbered) and must reply with **one number**. Numbers are far easier to parse reliably than free-text label names.

Input text is **PII-redacted** (same as production) but not letter-normalised, because an LLM reads natural text better than normalised text.

In [ ]:
from mizan.pii import redact

labels = json.load(open("data/processed/labels.json", encoding="utf-8"))["label2id"]
LABELS = sorted(labels, key=labels.get)          # index i -> label
menu = "\n".join(f"{i+1}. {lab}" for i, lab in enumerate(LABELS))

SYSTEM = "You are an intent classifier for a bank's Arabic customer-support channel."
def build_prompt(text: str) -> str:
    user = (
        "Classify the customer's message into exactly one of these banking intents.\n"
        f"{menu}\n\n"
        f"Customer message: {text}\n\n"
        "Reply with the intent number only (1-77)."
    )
    msgs = [{"role": "system", "content": SYSTEM}, {"role": "user", "content": user}]
    return tok.apply_chat_template(msgs, tokenize=False, add_generation_prompt=True)

def parse(output: str) -> str:
    m = re.search(r"\d+", output)
    if m and 1 <= int(m.group()) <= len(LABELS):
        return LABELS[int(m.group()) - 1]
    return "UNPARSEABLE"   # counted as wrong

test = pd.read_csv("data/processed/test.csv")
print(build_prompt(redact(test.text[0])[0])[-400:])

## 4. Run on the test set

In [ ]:
BATCH = 16
prompts = [build_prompt(redact(t)[0]) for t in test.text]
raw_outputs = []

torch.cuda.synchronize(); start = time.perf_counter()
for i in range(0, len(prompts), BATCH):
    enc = tok(prompts[i:i+BATCH], return_tensors="pt", padding=True).to(model.device)
    with torch.no_grad():
        out = model.generate(**enc, max_new_tokens=6, do_sample=False, pad_token_id=tok.eos_token_id)
    raw_outputs += tok.batch_decode(out[:, enc.input_ids.shape[1]:], skip_special_tokens=True)
torch.cuda.synchronize()
LATENCY_MS = (time.perf_counter() - start) * 1000 / len(prompts)

preds = [parse(o) for o in raw_outputs]
print(f"{LATENCY_MS:.0f} ms per query (batched on T4), unparseable: {preds.count('UNPARSEABLE')}")

## 5. Measure single-query latency

The batched figure above flatters the LLM. A support chatbot answers **one customer at a time**, so this measures that case on 30 queries.

In [ ]:
single = []
for t in test.text[:30]:
    enc = tok([build_prompt(redact(t)[0])], return_tensors="pt").to(model.device)
    torch.cuda.synchronize(); s = time.perf_counter()
    with torch.no_grad():
        model.generate(**enc, max_new_tokens=6, do_sample=False, pad_token_id=tok.eos_token_id)
    torch.cuda.synchronize(); single.append((time.perf_counter() - s) * 1000)
SINGLE_MS = sum(single) / len(single)
print(f"{SINGLE_MS:.0f} ms per query, one at a time (T4 GPU)")

## 6. Score and save

In [ ]:
from mizan.evaluate import evaluate

result = evaluate(
    "Zero-shot LLM (Qwen2.5-1.5B-Instruct)",
    test.label.tolist(), preds, test.variety.tolist(),
    latency_ms=SINGLE_MS, size_mb=SIZE_MB,
    notes=(f"no training; T4 GPU fp16; single-query latency {SINGLE_MS:.0f} ms "
           f"(batched {LATENCY_MS:.0f} ms); unparseable outputs: {preds.count('UNPARSEABLE')}"),
)
print(json.dumps(result, ensure_ascii=False, indent=2))

Look at some mistakes, which is useful for the README discussion:

In [ ]:
wrong = test.assign(pred=preds, raw=raw_outputs)[lambda d: d.label != d.pred]
wrong[["text", "label", "pred", "variety"]].head(15)

## 7. Download the result

Download `results/zero_shot_llm_qwen2_5_1_5b_instruct.json` and put it in your repository's `results/` folder, then run `python -m mizan.report --write` to update the README.

In [ ]:
from google.colab import files
files.download("results/zero_shot_llm_qwen2_5_1_5b_instruct.json")